# Байесовская статистика

## Модуль 4: Байесовский вывод, априорные и апостериорные распределения

В этом ноутбуке мы рассмотрим основы байесовской статистики и её применение.

### Содержание:
1. Теорема Байеса в контексте статистики
2. Априорные распределения
3. Функция правдоподобия
4. Апостериорные распределения
5. Сопряжённые априорные
6. Байесовский интервальный оценка (credible interval)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.integrate import quad
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Библиотеки успешно загружены!')

## 1. Теорема Байеса в контексте статистики

### Классическая теорема Байеса:
$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

### В контексте оценки параметров:
$$P(\theta | \text{data}) = \frac{P(\text{data} | \theta) \cdot P(\theta)}{P(\text{data})}$$

или:

$$\underbrace{P(\theta | \text{data})}_{\text{Апостериорное}} \propto \underbrace{P(\text{data} | \theta)}_{\text{Правдоподобие}} \cdot \underbrace{P(\theta)}_{\text{Априорное}}$$

### Компоненты:
- **$P(\theta)$ — априорное (prior):** Наши знания о параметре ДО наблюдения данных
- **$P(\text{data} | \theta)$ — правдоподобие (likelihood):** Вероятность данных при заданном параметре
- **$P(\theta | \text{data})$ — апостериорное (posterior):** Наши знания о параметре ПОСЛЕ наблюдения данных
- **$P(\text{data})$ — нормализующая константа (evidence):** Не зависит от $\theta$

### Байесовский подход vs Частотный:
- **Частотный:** $\theta$ — фиксированный неизвестный параметр
- **Байесовский:** $\theta$ — случайная величина с распределением

In [ ]:
# Демонстрация: Байесовское обновление
# Пример: Оценка вероятности успеха

# Априорное: Beta(2, 2) — слабое априорное
alpha_prior = 2
beta_prior = 2

# Данные: 7 успехов из 10 испытаний
n_trials = 10
n_successes = 7

# Апостериорное: Beta(α + successes, β + failures)
alpha_post = alpha_prior + n_successes
beta_post = beta_prior + (n_trials - n_successes)

print('Байесовское обновление')
print('=' * 60)
print(f'Априорное: Beta({alpha_prior}, {beta_prior})')
print(f'Данные: {n_successes} успехов из {n_trials}')
print(f'Апостериорное: Beta({alpha_post}, {beta_post})')

# Визуализация
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

p_range = np.linspace(0, 1, 1000)

# Априорное
axes[0].plot(p_range, stats.beta.pdf(p_range, alpha_prior, beta_prior), 
             'b-', linewidth=2, label=f'Beta({alpha_prior}, {beta_prior})')
axes[0].fill_between(p_range, 
                     stats.beta.pdf(p_range, alpha_prior, beta_prior), 
                     alpha=0.3)
axes[0].set_xlabel('p')
axes[0].set_ylabel('Плотность')
axes[0].set_title('Априорное распределение')
axes[0].legend()
axes[0].set_ylim(0, 3)

# Правдоподобие
likelihoods = [stats.binom.pmf(n_successes, n_trials, p) for p in p_range]
axes[1].plot(p_range, likelihoods, 'g-', linewidth=2)
axes[1].fill_between(p_range, likelihoods, alpha=0.3, color='green')
axes[1].axvline(n_successes/n_trials, color='red', linestyle='--', linewidth=2,
                label=f'MLE = {n_successes/n_trials:.2f}')
axes[1].set_xlabel('p')
axes[1].set_ylabel('L(p | data)')
axes[1].set_title('Правдоподобие')
axes[1].legend()

# Апостериорное
axes[2].plot(p_range, stats.beta.pdf(p_range, alpha_post, beta_post), 
             'r-', linewidth=2, label=f'Beta({alpha_post}, {beta_post})')
axes[2].fill_between(p_range, 
                     stats.beta.pdf(p_range, alpha_post, beta_post), 
                     alpha=0.3, color='red')

# MAP-оценка
p_map = (alpha_post - 1) / (alpha_post + beta_post - 2)
axes[2].axvline(p_map, color='blue', linestyle='--', linewidth=2,
                label=f'MAP = {p_map:.2f}')

axes[2].set_xlabel('p')
axes[2].set_ylabel('P(p | data)')
axes[2].set_title('Апостериорное распределение')
axes[2].legend()
axes[2].set_ylim(0, 3)

plt.tight_layout()
plt.show()

## 2. Сопряжённые априорные

**Сопряжённое априорное** — это априорное распределение, при котором апостериорное belongs to the same family.

| Правдоподобие | Сопряжённое априорное | Апостериорное |
|--------------|---------------------|---------------|
| Бернулли/Биномиальное | Beta($\alpha, \beta$) | Beta($\alpha + x, \beta + n - x$) |
| Пуассон | Gamma($\alpha, \beta$) | Gamma($\alpha + \sum x_i, \beta + n$) |
| Нормальное (известная $\sigma^2$) | N($\mu_0, \sigma_0^2$) | N($\mu_n, \sigma_n^2$) |
| Экспоненциальное | Gamma($\alpha, \beta$) | Gamma($\alpha + n, \beta + \sum x_i$) |

### Преимущества:
- Апостериорное можно вычислить аналитически
- Легко обновлять при поступлении новых данных
- Интерпретируемость параметров

In [ ]:
# Пример: Сопряжённые априорные для разных распределений

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Бернулли + Beta
alpha, beta = 2, 2  # Априорное
n, x = 20, 12  # Данные
alpha_post = alpha + x
beta_post = beta + n - x

p_range = np.linspace(0, 1, 1000)
axes[0, 0].plot(p_range, stats.beta.pdf(p_range, alpha, beta), 
                'b-', linewidth=2, label=f'Prior: Beta({alpha}, {beta})')
axes[0, 0].plot(p_range, stats.beta.pdf(p_range, alpha_post, beta_post), 
                'r-', linewidth=2, label=f'Posterior: Beta({alpha_post}, {beta_post})')
axes[0, 0].set_xlabel('p')
axes[0, 0].set_ylabel('Плотность')
axes[0, 0].set_title('Бернулли + Beta априорное')
axes[0, 0].legend()

# 2. Пуассон + Gamma
alpha, beta = 2, 1  # Априорное Gamma(shape, rate)
data = [3, 5, 2, 4, 6, 3, 5, 4]  # Данные
n = len(data)
sum_x = sum(data)
alpha_post = alpha + sum_x
beta_post = beta + n

x_range = np.linspace(0, 10, 1000)
axes[0, 1].plot(x_range, stats.gamma.pdf(x_range, alpha, scale=1/beta), 
                'b-', linewidth=2, label=f'Prior: Gamma({alpha}, {beta})')
axes[0, 1].plot(x_range, stats.gamma.pdf(x_range, alpha_post, scale=1/beta_post), 
                'r-', linewidth=2, label=f'Posterior: Gamma({alpha_post}, {beta_post})')
axes[0, 1].axvline(sum_x/n, color='green', linestyle='--', linewidth=2, 
                   label=f'MLE = {sum_x/n:.2f}')
axes[0, 1].set_xlabel('λ')
axes[0, 1].set_ylabel('Плотность')
axes[0, 1].set_title('Пуассон + Gamma априорное')
axes[0, 1].legend()

# 3. Нормальное (известная σ²) + Normal
mu_0, sigma_0 = 5, 2  # Априорное
sigma = 1  # Известная σ данных
data = np.random.normal(6, sigma, 10)  # Данные
n = len(data)
x_bar = np.mean(data)

# Апостериорное
sigma_0_sq = sigma_0**2
sigma_sq = sigma**2
mu_n = (n*x_bar/sigma_sq + mu_0/sigma_0_sq) / (n/sigma_sq + 1/sigma_0_sq)
sigma_n_sq = 1 / (n/sigma_sq + 1/sigma_0_sq)

x_range = np.linspace(2, 9, 1000)
axes[1, 0].plot(x_range, stats.norm.pdf(x_range, mu_0, sigma_0), 
                'b-', linewidth=2, label=f'Prior: N({mu_0}, {sigma_0}²)')
axes[1, 0].plot(x_range, stats.norm.pdf(x_range, mu_n, np.sqrt(sigma_n_sq)), 
                'r-', linewidth=2, label=f'Posterior: N({mu_n:.2f}, {np.sqrt(sigma_n_sq):.2f}²)')
axes[1, 0].axvline(x_bar, color='green', linestyle='--', linewidth=2, 
                   label=f'MLE = {x_bar:.2f}')
axes[1, 0].set_xlabel('μ')
axes[1, 0].set_ylabel('Плотность')
axes[1, 0].set_title('Нормальное + Normal априорное')
axes[1, 0].legend()

# 4. Байесовское обновление (последовательное)
# Начинаем с априорного и последовательно добавляем данные
alpha, beta = 1, 1  # Начальное априорное (равномерное)
data_sequence = [1, 0, 1, 1, 0, 1, 1, 1, 0, 1]  # Последовательность успехов

p_range = np.linspace(0, 1, 1000)
axes[1, 1].plot(p_range, stats.beta.pdf(p_range, alpha, beta), 
                'k--', linewidth=1, label='Prior')

for i, x in enumerate(data_sequence):
    alpha += x
    beta += (1 - x)
    if i in [0, 2, 4, 7, 9]:  # Показать только некоторые шаги
        axes[1, 1].plot(p_range, stats.beta.pdf(p_range, alpha, beta), 
                        linewidth=2, label=f'n={i+1}')

axes[1, 1].set_xlabel('p')
axes[1, 1].set_ylabel('Плотность')
axes[1, 1].set_title('Последовательное байесовское обновление')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## 3. Байесовский интервальный оценка (Credible Interval)

**Доверительный интервал (частотный):**
- Если повторить эксперимент много раз, то 95% интервалов содержат истинное $\theta$

**Доверительный интервал (байесовский, credible interval):**
- С вероятностью 95% истинное $\theta$ лежит в этом интервале

### Типы credible intervals:
1. **Центральный (equal-tailed):** 2.5% и 97.5% квантили апостериорного
2. **Highest Posterior Density (HPD):** Наименьший интервал, содержащий 95% апостериорного

$$\text{HPD}: \{\theta : P(\theta | \text{data}) \geq c\}$$

где $c$ выбрано так, чтобы $P(\text{HPD} | \text{data}) = 0.95$

In [ ]:
# Пример: Байесовский credible interval
np.random.seed(42)

# Данные
n = 20
x = 12  # Количество успехов

# Априорное: Beta(2, 2)
alpha_prior = 2
beta_prior = 2

# Апостериорное
alpha_post = alpha_prior + x
beta_post = beta_prior + n - x

# 95% credible interval (equal-tailed)
ci_lower = stats.beta.ppf(0.025, alpha_post, beta_post)
ci_upper = stats.beta.ppf(0.975, alpha_post, beta_post)

# 95% HPD interval (приближённо)
p_range = np.linspace(0, 1, 10000)
posterior = stats.beta.pdf(p_range, alpha_post, beta_post)

# Найти HPD
sorted_posterior = np.sort(posterior)[::-1]
cumulative = np.cumsum(sorted_posterior) / np.sum(sorted_posterior)
threshold_idx = np.searchsorted(cumulative, 0.95)
threshold = sorted_posterior[threshold_idx]

hpd_mask = posterior >= threshold
hpd_lower = p_range[hpd_mask][0]
hpd_upper = p_range[hpd_mask][-1]

print('Байесовский credible interval')
print('=' * 60)
print(f'Данные: {x} успехов из {n}')
print(f'Априорное: Beta({alpha_prior}, {beta_prior})')
print(f'Апостериорное: Beta({alpha_post}, {beta_post})')
print(f'\n95% Equal-tailed CI: [{ci_lower:.4f}, {ci_upper:.4f}]')
print(f'95% HPD interval: [{hpd_lower:.4f}, {hpd_upper:.4f}]')
print(f'\nMAP-оценка: {(alpha_post-1)/(alpha_post+beta_post-2):.4f}')
print(f'Среднее апостериорного: {alpha_post/(alpha_post+beta_post):.4f}')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Апостериорное с credible intervals
axes[0].plot(p_range, posterior, 'b-', linewidth=2, label='Апостериорное')

# Equal-tailed
mask_et = (p_range >= ci_lower) & (p_range <= ci_upper)
axes[0].fill_between(p_range[mask_et], posterior[mask_et], 
                     alpha=0.3, color='blue', label='95% Equal-tailed')

# HPD
axes[0].fill_between(p_range[hpd_mask], posterior[hpd_mask], 
                     alpha=0.3, color='red', label='95% HPD')

axes[0].set_xlabel('p')
axes[0].set_ylabel('Плотность')
axes[0].set_title('Байесовский credible interval')
axes[0].legend()

# Сравнение с частотным ДИ
# Частотный ДИ (Вальд)
p_hat = x / n
se = np.sqrt(p_hat * (1 - p_hat) / n)
freq_ci_lower = p_hat - 1.96 * se
freq_ci_upper = p_hat + 1.96 * se

print(f'\nСравнение:')
print(f'  Частотный 95% ДИ: [{freq_ci_lower:.4f}, {freq_ci_upper:.4f}]')
print(f'  Байесовский 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]')

# Визуализация сравнения
axes[1].plot(p_range, posterior, 'b-', linewidth=2, label='Апостериорное')
axes[1].axvspan(freq_ci_lower, freq_ci_upper, alpha=0.2, color='green', 
                label='Частотный ДИ')
axes[1].axvspan(ci_lower, ci_upper, alpha=0.2, color='blue', 
                label='Байесовский CI')
axes[1].axvline(p_hat, color='green', linestyle='--', linewidth=2, 
                label=f'p̂ = {p_hat:.2f}')
axes[1].set_xlabel('p')
axes[1].set_ylabel('Плотность')
axes[1].set_title('Сравнение частотного и байесовского интервалов')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Байесовский вывод для нормального распределения

### Задача:
Оценить $\mu$ нормального распределения с известной $\sigma^2$.

### Априорное:
$$\mu \sim N(\mu_0, \sigma_0^2)$$

### Правдоподобие:
$$X_1, ..., X_n | \mu \sim N(\mu, \sigma^2)$$

### Апостериорное:
$$\mu | X_1, ..., X_n \sim N(\mu_n, \sigma_n^2)$$

где:
$$\mu_n = \frac{n\bar{X}/\sigma^2 + \mu_0/\sigma_0^2}{n/\sigma^2 + 1/\sigma_0^2}$$
$$\frac{1}{\sigma_n^2} = \frac{n}{\sigma^2} + \frac{1}{\sigma_0^2}$$

### Интерпретация:
- Апостериорное среднее — взвешенное среднее между априорным и MLE
- Веса пропорциональны точностям (обратным дисперсиям)
- При $n \to \infty$: апостериорное стремится к MLE

In [ ]:
# Пример: Байесовский вывод для нормального распределения
np.random.seed(42)

# Параметры
mu_true = 5  # Истинное значение (неизвестное)
sigma = 1    # Известная σ данных

# Априорное
mu_0 = 3     # Априорное среднее
sigma_0 = 2  # Априорное стандартное отклонение

# Данные
n_samples = [1, 5, 10, 20, 50]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

# Априорное
x_range = np.linspace(-2, 10, 1000)
axes[0].plot(x_range, stats.norm.pdf(x_range, mu_0, sigma_0), 
             'b-', linewidth=2, label=f'Prior: N({mu_0}, {sigma_0}²)')
axes[0].axvline(mu_true, color='red', linestyle='--', linewidth=2, 
                label=f'Истинное μ = {mu_true}')
axes[0].set_xlabel('μ')
axes[0].set_ylabel('Плотность')
axes[0].set_title('Априорное распределение')
axes[0].legend()

# Обновление при разных размерах выборки
for idx, n in enumerate(n_samples[:5]):
    data = np.random.normal(mu_true, sigma, n)
    x_bar = np.mean(data)
    
    # Апостериорное
    precision_data = n / sigma**2
    precision_prior = 1 / sigma_0**2
    mu_n = (precision_data * x_bar + precision_prior * mu_0) / \
           (precision_data + precision_prior)
    sigma_n_sq = 1 / (precision_data + precision_prior)
    sigma_n = np.sqrt(sigma_n_sq)
    
    # Визуализация
    axes[idx+1].plot(x_range, stats.norm.pdf(x_range, mu_0, sigma_0), 
                     'b--', linewidth=1, alpha=0.5, label='Prior')
    axes[idx+1].plot(x_range, stats.norm.pdf(x_range, mu_n, sigma_n), 
                     'r-', linewidth=2, label=f'Posterior: N({mu_n:.2f}, {sigma_n:.2f}²)')
    axes[idx+1].axvline(mu_true, color='green', linestyle='--', linewidth=2, 
                        label=f'Истинное μ = {mu_true}')
    axes[idx+1].axvline(x_bar, color='orange', linestyle='--', linewidth=2, 
                        label=f'MLE = {x_bar:.2f}')
    axes[idx+1].set_xlabel('μ')
    axes[idx+1].set_ylabel('Плотность')
    axes[idx+1].set_title(f'n = {n}')
    axes[idx+1].legend(fontsize=8)

plt.tight_layout()
plt.show()

# Числовые результаты
print('Байесовское обновление для нормального распределения')
print('=' * 70)
print(f'Истинное μ = {mu_true}')
print(f'Априорное: N({mu_0}, {sigma_0}²)')
print(f'σ данных = {sigma}')
print()

for n in n_samples:
    data = np.random.normal(mu_true, sigma, n)
    x_bar = np.mean(data)
    
    precision_data = n / sigma**2
    precision_prior = 1 / sigma_0**2
    mu_n = (precision_data * x_bar + precision_prior * mu_0) / \
           (precision_data + precision_prior)
    sigma_n_sq = 1 / (precision_data + precision_prior)
    
    print(f'n = {n:>3}: X̄ = {x_bar:.3f}, μ_n = {mu_n:.3f}, σ_n² = {sigma_n_sq:.4f}')

## Упражнения

### Упражнение 1: Байесовское обновление для монеты
У вас есть монета, и вы хотите оценить вероятности орла $p$.

1. Начните с априорного Beta(1, 1) (равномерное)
2. После 10 бросков (6 орлов) найдите апостериорное
3. После ещё 10 бросков (7 орлов) обновите апостериорное
4. Постройте графики априорного и двух апостериорных

### Упражнение 2: Байесовский credible interval
Используя данные из упражнения 1:
1. Постройте 95% equal-tailed credible interval
2. Постройте 95% HPD interval
3. Сравните с частотным доверительным интервалом

### Упражнение 3: Нормальное априорное
Даны данные о времени доставки: [25, 30, 28, 32, 27, 29, 31, 26, 33, 28]

Используя априорное N(30, 5²) и известную σ = 3:
1. Найдите апостериорное распределение
2. Постройте 95% credible interval для среднего
3. Как изменится результат при априорном N(20, 1²)?

### Упражнение 4: Выбор априорного
Объясните, какое априорное вы бы выбрали в каждом случае:
1. Оценка вероятности редкой болезни (p ≈ 0.001)
2. Оценка среднего роста студентов (μ ≈ 170 см)
3. Оценка эффективности нового лекарства (нет информации)

---

**Решения** можно найти в ноутбуке `solutions/10_Solutions.ipynb`